# DevOps and data

CI logs, migrations, config diffs, and a few customer rows. Arithmetic stays in Python. Jev only labels.


In [ ]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


## 43. Why did CI fail?

A business-logic assertion in a file you changed is a regression. A full disk before tests start is infra.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "cause": Choice(
            instructions="What most likely caused the failure in `log_tail`?",
            criteria={
                "flaky_test": "Timing or a network blip",
                "regression": "The change broke real behavior",
                "infra": "Runner, disk, or environment",
                "dependency": "A package could not be fetched",
            },
        ),
        "related_to_diff": Noul(instructions="Does `log_tail` mention a file listed in `diff_files`?"),
    }
    for job in load_json("logs.json")["ci_jobs"]:
        response = ask(job, questions)
        show(response)
        cause = response.choices["cause"]
        if cause.choice == "flaky_test" and cause.confidence > 0.7 and response.nouls["related_to_diff"].noul < 0.3:
            route = "retry"
        elif cause.choice == "regression":
            route = "block_merge"
        else:
            route = "notify_platform"
        print(job["name"], "->", route)


**What you should see.** The payment assertion should block the merge. The full disk should notify platform.


## 44. Rescore a log line

The logged level is not always the real severity. Keep lines that land at warning or above.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "true_severity": Score(
            instructions="How severe is the event in `line`, ignoring any logged level word?",
            criteria=["Debug noise", "Informational", "Warning", "Error", "Critical"],
        )
    }
    lines = load_json("logs.json")["app_lines"]
    requests = [{"state": {"line": line}, "questions": questions} for line in lines]
    for line, response in zip(lines, ask_many(requests)):
        score = response.scores["true_severity"].score
        if score >= 2:
            print(round(score, 2), line)


**What you should see.** The payment 500 should be kept. The cache-hit debug line should not.


## 45. Is this migration safe to apply?

Adding a nullable column is not the same as dropping one.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "kind": Choice(
            instructions="What kind of change is `sql`?",
            criteria={
                "additive": "Adds a column, table, or index only",
                "breaking": "Drops, renames, or narrows a column",
                "cosmetic": "Comments or formatting only",
            },
        ),
        "locks_table": Noul(instructions="Would `sql` take a long lock or rewrite a table?"),
    }
    for item in load_json("logs.json")["migrations"]:
        response = ask(item, questions)
        show(response)
        if response.choices["kind"].choice == "breaking" or response.nouls["locks_table"].noul > 0.55:
            route = "dba_review"
        elif response.choices["kind"].confidence < 0.6:
            route = "dba_review"
        else:
            route = "auto_apply"
        print(item["name"], "->", route)


**What you should see.** `add_gift_note` can auto-apply. `drop_phone` needs a DBA.


## 46. Blast radius of a config diff

One Score per changed key, in a single call. The run's blast radius is the max.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    changes = load_json("logs.json")["config_changes"]
    questions = {
        "impact_%s" % i: Score(
            instructions="If changes[%s] is wrong, how bad is it?" % i,
            criteria=["Cosmetic", "One service degraded", "Security or data exposure"],
        )
        for i in range(len(changes))
    }
    response = ask({"changes": changes}, questions)
    show(response)
    worst = max(response.scores["impact_%s" % i].score for i in range(len(changes)))
    print("blast radius:", round(worst, 2))
    print("route:", "human_review" if worst >= 1.5 else "auto_apply")


**What you should see.** The tagline is cosmetic. Replacing a live payment key should push the max into human review.


## 47. Spot-check a few customer rows

Ask simple quality questions. The share of rows that fail is computed in Python.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    sample = [row for row in customers() if row["id"] in ("C01", "C11", "C12", "C08")]
    questions = {
        "name_is_person": Noul(instructions="Is `name` a person's name rather than a company, a team, or a test placeholder?"),
        "email_plausible": Noul(instructions="Does `email` look like a real email address rather than a test value?"),
    }
    requests = [{"state": {"row": row}, "questions": questions} for row in sample]
    results = ask_many(requests)
    for name in questions:
        share = sum(item.nouls[name].noul > 0.5 for item in results) / len(results)
        print(name, "share passing", round(share, 2))
    for row, response in zip(sample, results):
        print(row["id"], row["name"], "person", round(response.nouls["name_is_person"].noul, 2))


**What you should see.** Maya and Jonah should look like people. Test User should not. Acme Holdings is a company, so `name_is_person` should be low.


## 48. Which runbook section matches the alert?

The sections are the headings in `incident-runbook.md`. `none` is a real option.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    text = read_text("incident-runbook.md")
    sections = {}
    current = None
    bucket = []
    for line in text.splitlines():
        if line.startswith("## "):
            if current:
                sections[current] = " ".join(bucket)
            current = line[3:]
            bucket = []
        elif current:
            bucket.append(line)
    if current:
        sections[current] = " ".join(bucket)
    sections["none"] = "No section applies"
    for alert in load_json("alerts.json"):
        response = ask(
            {"alert": alert["text"]},
            {"section": Choice(instructions="Which runbook section applies to `alert`?", criteria=sections)},
        )
        show(response)
        answer = response.choices["section"]
        print(alert["id"], "->", None if answer.choice == "none" or answer.confidence < 0.5 else answer.choice)


**What you should see.** The checkout 500 should match Checkout 500s. The printer jam should match Label printer.
